In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as mpl
import seaborn as sb
import missingno as msn
import warnings
warnings.filterwarnings('ignore')

print('OK')

OK


In [3]:
train = pd.read_parquet('C:/Users/Deepayan/Documents/MAVERICK/projects/churn-prediction/artifacts/kkbox_final.parquet')
print('OK')

OK


In [4]:
train.shape, train.dtypes, train.memory_usage(deep=True).sum() / 1e6

((970960, 19),
 msno                          str
 is_churn                    int64
 city                      float64
 bd                        float64
 gender                        str
 registered_via            float64
 registration_init_time    float64
 transanction_count        float64
 first_transaction_date    float64
 last_transaction_date     float64
 last_plan_days            float64
 last_plan_price           float64
 avg_amount_paid           float64
 auto_renew_flag           float64
 cancel_count              float64
 total_secs                float64
 num_unq                   float64
 num_100                   float64
 days_active               float64
 dtype: object,
 np.float64(192.35397))

In [5]:
train['msno'].duplicated().sum()

np.int64(0)

In [6]:
train['city'].isnull().sum() #pandas treats NaN values by upcasting them to float64, so check for null values

np.int64(109993)

In [7]:
train['registered_via'].isnull().sum() #pandas treats NaN values by upcasting them to float64, so check for null values

np.int64(109993)

In [8]:
whether_city_registered_via_have_same_missing_rows = train.loc[train['city'].isnull(), 'msno'].equals(train.loc[train['registered_via'].isnull(), 'msno'])
print(whether_city_registered_via_have_same_missing_rows) 

#checking is the same rows have both city and registered_via missing
#if that is true, it confirms this is not a merge bug

True


In [9]:
train['has_details'] = train['city'].notnull()
pd.crosstab(train['has_details'], train['is_churn'], normalize= 'index') 

#relates the is_churn column with 'city' and 'registered_via' when they are null and not null

is_churn,0,1
has_details,,
False,0.946524,0.053476
True,0.905399,0.094601


-> Users who have signed up and completed their profile (filled in their 'city' and 'registered_via' info in details) have churned at 5.3%
-> Users who have signed up but not completed their profile (filled in their 'city' and 'registered_via' info in details) have churned at 9.5%

Hence, profile incompleteness does not directly lead to user being churned as nearly double the users have actually churned who have complete profiles than those who have incomplete profiles. 

In [10]:
train.groupby('has_details')['registration_init_time'].describe()

,count,mean,std,min,25%,50%,75%,max
has_details,,,,,,,,
False,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
True,860967.0,2.013265e+07,30111.744263,20040326.0,20120214.0,20140602.0,20160118.0,20170424.0


This confirms that all users who have 'city' and 'registered_via' details missing also have 'registration_init_time' blank, strongly leading to the conclusion that these users ('msno') do not exist in the members.csv file but since they exist in the merged parquet, they must have records in transactions.csv and user_logs.csv

In [11]:
train.groupby('has_details')['first_transaction_date'].describe()

,count,mean,std,min,25%,50%,75%,max
has_details,,,,,,,,
False,108210.0,2.017027e+07,947.033560,20150102.0,20170308.0,20170316.0,20170324.0,20170331.0
True,825368.0,2.017011e+07,1838.337256,20150101.0,20170306.0,20170315.0,20170323.0,20170331.0


COUNT

Out of 109993 users with no details, 108210 users have a transaction date in the window which means there are 1783 users with no details who also have zero transactions in this window.

->The mean, median, 25, 75 percentiles all have dates having a gap of nearly 1-2 days for whether the person has details or not. This effectively rules out the assumption that there must have been a legacy account which logged in users during earlier times without details who have somehow managed to not churn till now before introducing a modified subscription plan for new users requiring their details who churn at a faster rate.

MEDIAN (50%)

The min column shows that earliest transactions date back to 2015-01-02 but the median column shows 2017-03-16, much closer to the max column showing 2017-03-31. This effectively means that there have been significantly more transactions in the last 2-3 weeks of a 2 year window.

In [17]:
print(train['first_transaction_date'].min(), train['last_transaction_date'].max())

train['has_transactions'] = train['transanction_count'].notnull()
pd.crosstab(train['has_transactions'], train['is_churn'], normalize = 'index')


20150101.0 20170331.0


is_churn,0,1
has_transactions,,
False,0.212589,0.787411
True,0.937986,0.062014


The full range of the recorded transactions in the dataset is from 1st January, 2015 to 31st March, 2017.

Crosstabing whether a user has transactions with their respective churn rates highlights two thing:

1. 78.7% of users who don't have any transactions in this window have churned.
2. Only 6.2% of users who have actually recorded transactions have churned.

Checking whether churned vs non-churned users' last transaction dates diverge in a way consistent with label leakage from the renewal-check window

In [21]:
train['last_transaction_date_dt'] = pd.to_datetime(train['last_transaction_date'], format= '%Y%m%d')
train.groupby('is_churn')['last_transaction_date_dt'].describe()

,count,mean,min,25%,50%,75%,max
is_churn,,,,,,,
0,875683,2017-03-17 22:24:55.930376,2015-11-30 00:00:00,2017-03-09 00:00:00,2017-03-18 00:00:00,2017-03-27 00:00:00,2017-03-31 00:00:00
1,57895,2017-02-23 18:01:44.091890,2015-01-02 00:00:00,2017-03-05 00:00:00,2017-03-13 00:00:00,2017-03-22 00:00:00,2017-03-31 00:00:00


1. The train dataset description lists February 2017 as the month when user subscriptions expire. A 30-day renewal period goes into March 2017.

2. From the description, the median and 75th percentiles show that most of the users who have at least one transaction have their last transaction date, roughly the same without much difference for churned and non-churned user (only a 5-day gap).

So the earlier narrative that no transactions at all = churn provides superficial information only. So the criteria should not be 'has_transactions', instead it should be 'has_transactions_after_feb_2017'.


In [18]:
train.head()

,msno,is_churn,city,bd,gender,registered_via,registration_init_time,transanction_count,first_transaction_date,last_transaction_date,...,last_plan_price,avg_amount_paid,auto_renew_flag,cancel_count,total_secs,num_unq,num_100,days_active,has_details,has_transactions
0,ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=,1,5.0,28.0,male,3.0,20131223.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,80598.557,348.0,318.0,11.0,True,False
1,f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=,1,13.0,20.0,male,3.0,20131223.0,1.0,20170311.0,20170311.0,...,180.0,180.0,0.0,0.0,6986.509,30.0,26.0,6.0,True,True
2,zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=,1,13.0,18.0,male,3.0,20131227.0,2.0,20170311.0,20170314.0,...,300.0,150.0,0.0,0.0,67810.467,432.0,205.0,20.0,True,True
3,8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=,1,1.0,0.0,NaN,7.0,20140109.0,10.0,20150808.0,20151208.0,...,149.0,149.0,1.0,0.0,NaN,NaN,NaN,NaN,True,True
4,K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=,1,13.0,35.0,female,7.0,20140125.0,8.0,20161001.0,20170316.0,...,99.0,99.0,1.0,1.0,239882.241,548.0,962.0,15.0,True,True
